In [1]:
import pandas as pd
import numpy as np
import joblib
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv("../data/processed/movies_clean.csv")
ratings = pd.read_csv("../data/processed/ratings_clean.csv")

tfidf_matrix = joblib.load("../models/tfidf_matrix.pkl")
movie_indices = joblib.load("../models/movie_indices.pkl")

user_movie_sparse = load_npz("../models/user_movie_sparse.npz")
movie_to_index = joblib.load("../models/movie_to_index.pkl")
index_to_movie = joblib.load("../models/index_to_movie.pkl")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)

Movies: (58098, 3)
Ratings: (27753444, 4)


In [3]:
def get_user_liked_movies(user_id, min_rating=4.0):
    user_ratings = ratings[
        (ratings["userId"] == user_id) &
        (ratings["rating"] >= min_rating)
    ]

    return set(user_ratings["movieId"])

In [4]:
user_id = ratings["userId"].iloc[0]

liked_movies = get_user_liked_movies(user_id)

print("User:", user_id)
print("Liked movies:", len(liked_movies))

User: 1
Liked movies: 6


In [5]:
def precision_at_k(recommended_movies, relevant_movies, k=10):

    recommended_movies = recommended_movies[:k]

    if len(recommended_movies) == 0:
        return 0

    relevant_count = sum(
        movie_id in relevant_movies
        for movie_id in recommended_movies
    )

    return relevant_count / len(recommended_movies)

In [6]:
def recall_at_k(recommended_movies, relevant_movies, k=10):

    recommended_movies = recommended_movies[:k]

    if len(relevant_movies) == 0:
        return 0

    relevant_count = sum(
        movie_id in relevant_movies
        for movie_id in recommended_movies
    )

    return relevant_count / len(relevant_movies)

In [7]:
sample_users = ratings["userId"].value_counts().head(20).index

print("Testing users:", len(sample_users))

Testing users: 20


In [8]:
precision_scores = []
recall_scores = []

for user_id in sample_users:

    user_ratings = ratings[
        ratings["userId"] == user_id
    ]

    liked_movies = set(
        user_ratings[
            user_ratings["rating"] >= 4
        ]["movieId"]
    )

    if len(liked_movies) < 2:
        continue

    # Use one liked movie as the query
    test_movie = list(liked_movies)[0]

    if test_movie not in movie_to_index:
        continue

    movie_index = movie_to_index[test_movie]

    scores = cosine_similarity(
        user_movie_sparse[:, movie_index].T,
        user_movie_sparse.T
    ).flatten()

    top_indices = scores.argsort()[-11:][::-1]

    recommended_ids = [
        index_to_movie[i]
        for i in top_indices
        if index_to_movie[i] != test_movie
    ][:10]

    precision_scores.append(
        precision_at_k(
            recommended_ids,
            liked_movies,
            10
        )
    )

    recall_scores.append(
        recall_at_k(
            recommended_ids,
            liked_movies,
            10
        )
    )

In [9]:
print(
    "Average Precision@10:",
    round(np.mean(precision_scores), 4)
)

print(
    "Average Recall@10:",
    round(np.mean(recall_scores), 4)
)

Average Precision@10: 0.41
Average Recall@10: 0.0032


In [10]:
evaluation_results = pd.DataFrame({
    "Metric": [
        "Precision@10",
        "Recall@10"
    ],
    "Score": [
        np.mean(precision_scores),
        np.mean(recall_scores)
    ]
})

evaluation_results

,Metric,Score
0,Precision@10,0.410000
1,Recall@10,0.003169


In [11]:
evaluation_results.to_csv(
    "../outputs/evaluation_results.csv",
    index=False
)

print("Evaluation results saved!")

Evaluation results saved!


In [13]:
from src.hybrid_recommender import HybridRecommender

recommender = HybridRecommender()

result = recommender.recommend(
    "Toy Story (1995)",
    10
)

result

ModuleNotFoundError: No module named 'src'